Before running this file, the processed_data file folder should be eltirely up to date. This is completed by running file_processing_spu.ipynb.


The goal of this file is to identify the location of the peaks in terms of adc channel of each file in processed_data. This ADC information is then printed to a .csv file for purposed of processing in different .ipynb files in this folder.

Because of the complexity of automating the spectrum, each spectrum will be identified manually.

This file:
- computes the background rate
- computes the rate of each target file
- subtracts the background rate from the target file's rate
- takes this adjusted data and creates a graph
- (creates a graph of unadjusted data on top of adjusted data to show effects of the background on data shape; this is unrelated to the propose of the data analysis but is important for the lab writeup)
- asks the user to identify peaks and ranges to fit gaussians to each
- extracts the center for the fitted gaussian as well as error data
- graphs the adjusted data with these fitted gaussians printed on top
- prints all of this extracted information to a .csv
- does this for (almost) every file in processed_data

In [1]:
# -----------------------------
# IMPORTS
# -----------------------------

import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import curve_fit

import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 6)

In [4]:
# -----------------------------
# PLOTLY / WIDGET DIAGNOSTIC
# -----------------------------

print("plotly version:", plotly.__version__)

try:
    import anywidget
    print("anywidget version:", anywidget.__version__)
except Exception as e:
    print("anywidget import failed:", repr(e))

try:
    import ipywidgets
    print("ipywidgets version:", ipywidgets.__version__)
except Exception as e:
    print("ipywidgets import failed:", repr(e))

import plotly
import plotly.graph_objects as go
import anywidget
import ipywidgets as widgets

print("plotly:", plotly.__version__)
print("anywidget:", anywidget.__version__)
print("ipywidgets:", widgets.__version__)

fig = go.FigureWidget(
    data=[go.Scatter(x=[1,2,3], y=[1,4,9], mode="lines+markers")]
)
fig

clicked = []

trace = fig.data[0]

def handle_click(trace, points, selector):
    if points.point_inds:
        i = points.point_inds[0]
        clicked.append((trace.x[i], trace.y[i]))
        print("clicked:", clicked[-1])

trace.on_click(handle_click)

fig

plotly version: 6.5.0
anywidget version: 0.9.21
ipywidgets version: 8.1.8
plotly: 6.5.0
anywidget: 0.9.21
ipywidgets: 8.1.8


FigureWidget({
    'data': [{'mode': 'lines+markers',
              'type': 'scatter',
              'uid': 'd6c8b2a9-f157-443a-968f-d68453edb6da',
              'x': [1, 2, 3],
              'y': [1, 4, 9]}],
    'layout': {'template': '...'}
})

In [5]:
clicked

[]

In [6]:
# -----------------------------
# FILE LOCATIONS
# -----------------------------

ROOT = Path("processed_data")
BACKGROUND_FILE = ROOT / "calibration" / "background_550PM_feb26_213PM_mar3.csv"

irrelevant = [
    "_single_test_converted.csv",
    "calibration/TESTFILE_FOR_CONVERSION_BGND.csv",
    "calibration/background_550PM_feb26_213PM_mar3.csv",
    "cs gain setting/Cs_possible_calcurve_feb_24.csv",
    "cs gain setting/Cs_trial_feb_19.csv",
    "calibration_curves/daily_calibration_params.csv"
]

irrelevant_set = set(irrelevant)

def normalize_relpath(path, root=ROOT):
    path = Path(path)
    try:
        rel = path.relative_to(root)
    except ValueError:
        rel = path
    return rel.as_posix()

print("ROOT:", ROOT)
print("BACKGROUND_FILE:", BACKGROUND_FILE)
print("Background exists:", BACKGROUND_FILE.exists())

ROOT: processed_data
BACKGROUND_FILE: processed_data/calibration/background_550PM_feb26_213PM_mar3.csv
Background exists: True


In [7]:
# -----------------------------
# FIT MODEL
# -----------------------------

def gaussian_with_background(x, A, mu, sigma, m, b):
    sigma = abs(sigma)
    return A * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2)) + m * x + b

In [8]:
# -----------------------------
# LOAD SPECTRUM / METADATA
# -----------------------------

def _extract_first_number(text):
    if text is None:
        return None

    match = re.search(r"[-+]?\d*\.?\d+", str(text))
    if match is None:
        return None

    try:
        return float(match.group())
    except ValueError:
        return None


def load_spectrum(csv_path):
    """
    Load a processed spectrum csv with preserved metadata and channel data.

    Expected processed structure:
    - metadata lines near the top
    - a line containing 'Channel Data:'
    - then a data table where:
        column 0 = channel / adc
        column 2 = counts
    """

    csv_path = Path(csv_path)

    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    metadata = {}
    data_start_idx = None

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "Channel Data:":
            data_start_idx = i + 1
            break

        if stripped.startswith("Elapsed Real Time:"):
            parts = stripped.split(",", 1)
            metadata["Elapsed Real Time"] = parts[1].strip() if len(parts) > 1 else ""
            continue

        if "Elapsed Real Time:" in stripped:
            value = stripped.split("Elapsed Real Time:", 1)[1]
            value = value.split(",", 1)[1].strip() if "," in value else value.strip()
            metadata["Elapsed Real Time"] = value
            continue

        if stripped.startswith("Start Time:"):
            metadata["Start Time"] = stripped.split("Start Time:", 1)[1].lstrip(",").strip()
            continue

        if stripped.startswith("End Time:"):
            metadata["End Time"] = stripped.split("End Time:", 1)[1].lstrip(",").strip()
            continue

        if "," in stripped:
            left, right = stripped.split(",", 1)
            left = left.strip().rstrip(":")
            right = right.strip()
            if left and right and left not in metadata:
                metadata[left] = right

    if data_start_idx is None:
        raise RuntimeError(f"Could not locate 'Channel Data:' block in {csv_path}")

    if data_start_idx < len(lines):
        header_line = lines[data_start_idx].strip().lower()
        if header_line.startswith("channel"):
            data_start_idx += 1

    numeric_rows = []

    for line in lines[data_start_idx:]:
        stripped = line.strip()

        if not stripped:
            continue

        parts = [p.strip() for p in stripped.split(",")]

        if len(parts) < 3:
            continue

        try:
            channel = float(parts[0])
            counts = float(parts[2])
            numeric_rows.append((channel, counts))
        except ValueError:
            continue

    if len(numeric_rows) == 0:
        raise RuntimeError(f"No numeric channel/count data found in {csv_path}")

    arr = np.array(numeric_rows, dtype=float)
    x = arr[:, 0]
    counts = arr[:, 1]

    elapsed_real_time = _extract_first_number(metadata.get("Elapsed Real Time"))
    metadata["Elapsed Real Time (s)"] = elapsed_real_time

    return x, counts, metadata


def counts_to_rate(counts, elapsed_time_seconds):
    if elapsed_time_seconds is None or elapsed_time_seconds <= 0:
        raise RuntimeError("Elapsed real time is missing or nonpositive.")
    return counts / elapsed_time_seconds


def compute_background_subtracted_rate(sample_path, background_path=BACKGROUND_FILE):
    """
    Background subtraction happens BEFORE any peak fitting.
    """

    x_sample, counts_sample, meta_sample = load_spectrum(sample_path)
    x_bg, counts_bg, meta_bg = load_spectrum(background_path)

    if len(x_sample) != len(x_bg) or not np.allclose(x_sample, x_bg):
        raise RuntimeError(
            "Sample and background channel grids do not match. "
            "Background subtraction cannot be performed safely."
        )

    t_sample = meta_sample.get("Elapsed Real Time (s)")
    t_bg = meta_bg.get("Elapsed Real Time (s)")

    raw_rate = counts_to_rate(counts_sample, t_sample)
    background_rate = counts_to_rate(counts_bg, t_bg)
    adjusted_rate = raw_rate - background_rate

    return x_sample, raw_rate, background_rate, adjusted_rate, meta_sample, meta_bg

In [9]:
# -----------------------------
# FILE LISTING
# -----------------------------

def get_candidate_files(root=ROOT):
    csv_files = sorted(root.rglob("*.csv"))
    kept = []

    for file in csv_files:
        rel = normalize_relpath(file, root)
        if rel in irrelevant_set:
            continue
        kept.append(file)

    return kept


candidate_files = get_candidate_files()

print("Number of candidate files:", len(candidate_files))
for file in candidate_files[:20]:
    print(" -", normalize_relpath(file))

Number of candidate files: 40
 - calibration/daily/Co60_21.9C_preamp100_mar5.csv
 - calibration/daily/Co60_22.4C_preamp100_feb26.csv
 - calibration/daily/Co60_22.5C_preamp100_mar3.csv
 - calibration/daily/Cs_21.9C_preamp0_mar5.csv
 - calibration/daily/Cs_22.0C_preamp0_mar3.csv
 - calibration/daily/Cs_22.4C_preamp0_feb26.csv
 - calibration/feb_24/preamp_0/Ba_22.4C_preamp0_feb24.csv
 - calibration/feb_24/preamp_0/Cs_22.1C_preamp0_feb24.csv
 - calibration/feb_24/preamp_100/Co60_22.5C_preamp100_feb24.csv
 - calibration/feb_24/preamp_100/Na_22.5C_preamp100_feb24.csv
 - data/Co57_runs_overnight/Co57_22.2C_preamp0_feb25_overnight.csv
 - data/feb_24/Na22/Na_22.5C_feb24_run1.csv
 - data/feb_24/co60/.ipynb_checkpoints/Co_22.4C_feb24_run1-checkpoint.csv
 - data/feb_24/co60/Co_22.4C_feb24_run1.csv
 - data/feb_26/co60/Co_22.8C_feb26_run1.csv
 - data/feb_26/cs137/Cs_22.4C_feb26_run1.csv
 - data/feb_26/na22/Na_22.6C_feb26_run1.csv
 - data/mar_3/co60/Co_22.5C_mar3_run1.csv
 - data/mar_3/cs137/Cs_22.5C

In [10]:
# -----------------------------
# CLICK DEBUG WITH PLOTLY
# -----------------------------

debug_state = {
    "clicked_x": [],
    "clicked_y": [],
    "fig": None,
    "trace": None,
    "out": None
}

def launch_plotly_click_debug():
    """
    Minimal click test. Use this BEFORE doing any data analysis.
    """

    x = np.linspace(0, 10, 500)
    y = np.sin(x)

    debug_state["clicked_x"] = []
    debug_state["clicked_y"] = []

    fig = go.FigureWidget(
        data=[
            go.Scatter(
                x=x,
                y=y,
                mode="lines",
                name="sin(x)"
            )
        ]
    )

    fig.update_layout(
        title="Plotly click debug: click the curve",
        xaxis_title="x",
        yaxis_title="sin(x)",
        clickmode="event+select",
        height=500
    )

    out = widgets.Output()

    trace = fig.data[0]

    def handle_click(trace, points, selector):
        if not points.point_inds:
            return

        idx = points.point_inds[0]
        x0 = float(x[idx])
        y0 = float(y[idx])

        debug_state["clicked_x"].append(x0)
        debug_state["clicked_y"].append(y0)

        with fig.batch_update():
            current_shapes = list(fig.layout.shapes) if fig.layout.shapes else []
            current_shapes.append(
                dict(
                    type="line",
                    x0=x0,
                    x1=x0,
                    y0=min(y),
                    y1=max(y),
                    line=dict(dash="dash")
                )
            )
            fig.layout.shapes = tuple(current_shapes)
            fig.layout.title = f"Last click: x={x0:.3f}, y={y0:.3f}"

        with out:
            print(f"Registered click at x={x0:.3f}, y={y0:.3f}")

    trace.on_click(handle_click)

    reset_button = widgets.Button(description="Reset clicks")
    check_button = widgets.Button(description="Show stored clicks")

    def reset_clicked(b):
        debug_state["clicked_x"] = []
        debug_state["clicked_y"] = []
        with fig.batch_update():
            fig.layout.shapes = ()
            fig.layout.title = "Plotly click debug: click the curve"
        with out:
            print("Clicks reset.")

    def show_clicked(b):
        with out:
            print("clicked_x =", debug_state["clicked_x"])
            print("clicked_y =", debug_state["clicked_y"])

    reset_button.on_click(reset_clicked)
    check_button.on_click(show_clicked)

    debug_state["fig"] = fig
    debug_state["trace"] = trace
    debug_state["out"] = out

    display(widgets.VBox([fig, widgets.HBox([reset_button, check_button]), out]))

In [11]:
# -----------------------------
# RUN CLICK DEBUG
# -----------------------------

launch_plotly_click_debug()

    'data': [{'mode': 'lines',
              'name': 'sin(x)',
              'ty…

In [12]:
# -----------------------------
# INSPECT CLICK DEBUG
# -----------------------------

print("clicked_x =", debug_state["clicked_x"])
print("clicked_y =", debug_state["clicked_y"])
print("number of clicks =", len(debug_state["clicked_x"]))

clicked_x = []
clicked_y = []
number of clicks = 0


In [13]:
# -----------------------------
# CURRENT FILE SELECTION
# -----------------------------

for i, file in enumerate(candidate_files):
    print(f"{i:2d} : {normalize_relpath(file)}")

 0 : calibration/daily/Co60_21.9C_preamp100_mar5.csv
 1 : calibration/daily/Co60_22.4C_preamp100_feb26.csv
 2 : calibration/daily/Co60_22.5C_preamp100_mar3.csv
 3 : calibration/daily/Cs_21.9C_preamp0_mar5.csv
 4 : calibration/daily/Cs_22.0C_preamp0_mar3.csv
 5 : calibration/daily/Cs_22.4C_preamp0_feb26.csv
 6 : calibration/feb_24/preamp_0/Ba_22.4C_preamp0_feb24.csv
 7 : calibration/feb_24/preamp_0/Cs_22.1C_preamp0_feb24.csv
 8 : calibration/feb_24/preamp_100/Co60_22.5C_preamp100_feb24.csv
 9 : calibration/feb_24/preamp_100/Na_22.5C_preamp100_feb24.csv
10 : data/Co57_runs_overnight/Co57_22.2C_preamp0_feb25_overnight.csv
11 : data/feb_24/Na22/Na_22.5C_feb24_run1.csv
12 : data/feb_24/co60/.ipynb_checkpoints/Co_22.4C_feb24_run1-checkpoint.csv
13 : data/feb_24/co60/Co_22.4C_feb24_run1.csv
14 : data/feb_26/co60/Co_22.8C_feb26_run1.csv
15 : data/feb_26/cs137/Cs_22.4C_feb26_run1.csv
16 : data/feb_26/na22/Na_22.6C_feb26_run1.csv
17 : data/mar_3/co60/Co_22.5C_mar3_run1.csv
18 : data/mar_3/cs137/

In [ ]:
# -----------------------------
# LOAD CURRENT FILE
# -----------------------------

file_index = int(input("Enter file index from the list above: "))

current_file = candidate_files[file_index]
current_rel = normalize_relpath(current_file)

x_current, raw_rate_current, background_rate_current, adjusted_rate_current, sample_meta_current, background_meta_current = (
    compute_background_subtracted_rate(current_file, background_path=BACKGROUND_FILE)
)

print("Current file:", current_rel)
print("Sample elapsed time (s):", sample_meta_current.get("Elapsed Real Time (s)"))
print("Background elapsed time (s):", background_meta_current.get("Elapsed Real Time (s)"))

In [ ]:
# -----------------------------
# RAW / BACKGROUND / ADJUSTED PLOT
# -----------------------------

plt.figure()
plt.plot(x_current, raw_rate_current, label="Raw sample rate")
plt.plot(x_current, background_rate_current, label="Background rate")
plt.plot(x_current, adjusted_rate_current, label="Background-subtracted rate")
plt.xlabel("ADC Channel")
plt.ylabel("Rate (counts/s)")
plt.title(f"{current_rel}\nRaw vs background vs background-subtracted")
plt.legend()
plt.show()

In [ ]:
# -----------------------------
# PLOTLY PEAK PICKER
# -----------------------------

peak_picker_state = {
    "clicked_centers": [],
    "fig": None,
    "trace": None,
    "out": None
}

def launch_peak_picker(x, y, title="Click peak centers"):
    """
    Plotly-based peak center picker.
    Click once per peak center.
    """

    peak_picker_state["clicked_centers"] = []

    fig = go.FigureWidget(
        data=[
            go.Scatter(
                x=x,
                y=y,
                mode="lines",
                name="Background-subtracted rate"
            )
        ]
    )

    fig.update_layout(
        title=title,
        xaxis_title="ADC Channel",
        yaxis_title="Background-subtracted rate (counts/s)",
        clickmode="event+select",
        height=550
    )

    out = widgets.Output()
    trace = fig.data[0]

    y_min = float(np.min(y))
    y_max = float(np.max(y))

    def handle_click(trace, points, selector):
        if not points.point_inds:
            return

        idx = int(points.point_inds[0])
        x0 = float(x[idx])

        peak_picker_state["clicked_centers"].append(x0)

        with fig.batch_update():
            current_shapes = list(fig.layout.shapes) if fig.layout.shapes else []
            current_shapes.append(
                dict(
                    type="line",
                    x0=x0,
                    x1=x0,
                    y0=y_min,
                    y1=y_max,
                    line=dict(dash="dash")
                )
            )
            fig.layout.shapes = tuple(current_shapes)
            fig.layout.title = f"{title} | last selected center = {x0:.3f}"

        with out:
            print(f"Registered peak center at ADC = {x0:.3f}")

    trace.on_click(handle_click)

    reset_button = widgets.Button(description="Reset centers")
    show_button = widgets.Button(description="Show centers")

    def reset_centers(b):
        peak_picker_state["clicked_centers"] = []
        with fig.batch_update():
            fig.layout.shapes = ()
            fig.layout.title = title
        with out:
            print("Peak centers reset.")

    def show_centers(b):
        with out:
            print("clicked_centers =", peak_picker_state["clicked_centers"])

    reset_button.on_click(reset_centers)
    show_button.on_click(show_centers)

    peak_picker_state["fig"] = fig
    peak_picker_state["trace"] = trace
    peak_picker_state["out"] = out

    display(widgets.VBox([fig, widgets.HBox([reset_button, show_button]), out]))

In [ ]:
# -----------------------------
# RUN PEAK PICKER FOR CURRENT FILE
# -----------------------------

launch_peak_picker(
    x_current,
    adjusted_rate_current,
    title=f"{current_rel} | click one time for each peak center"
)

In [ ]:
# -----------------------------
# BUILD PEAK PARAMETERS
# -----------------------------

def build_peak_params_from_plotly_clicks(clicked_centers):
    """
    For each clicked center, ask manually for the fit window.
    """
    peak_params = []

    if len(clicked_centers) == 0:
        print("No clicked peak centers found.")
        return peak_params

    for i, center in enumerate(clicked_centers, start=1):
        print(f"\nPeak {i}")
        print(f"Clicked center: {center:.3f}")

        xmin = float(input("Fit range minimum: "))
        xmax = float(input("Fit range maximum: "))

        if xmax < xmin:
            xmin, xmax = xmax, xmin

        peak_params.append((float(center), float(xmin), float(xmax)))

    return peak_params


def manual_peak_selection_typed(x, y):
    """
    Fully typed fallback.
    """
    plt.figure()
    plt.plot(x, y)
    plt.xlabel("ADC Channel")
    plt.ylabel("Background-subtracted rate (counts/s)")
    plt.title("Background-subtracted gamma spectrum")
    plt.show()

    n_peaks = int(input("Number of peaks to fit: "))

    peak_params = []

    for i in range(n_peaks):
        print(f"\nPeak {i+1}")
        center = float(input("Approximate peak center (ADC): "))
        xmin = float(input("Fit range minimum: "))
        xmax = float(input("Fit range maximum: "))

        if xmax < xmin:
            xmin, xmax = xmax, xmin

        peak_params.append((center, xmin, xmax))

    return peak_params

In [ ]:
# -----------------------------
# FINALIZE PEAK PARAMETERS
# -----------------------------

use_clicked = input("Use clicked peak centers from the Plotly picker? (y/n): ").strip().lower()

if use_clicked == "y":
    peak_params_current = build_peak_params_from_plotly_clicks(peak_picker_state["clicked_centers"])
else:
    peak_params_current = manual_peak_selection_typed(x_current, adjusted_rate_current)

print("peak_params_current =", peak_params_current)

In [ ]:
# -----------------------------
# FIT PEAKS
# -----------------------------

def fit_peaks(x, adjusted_rate, peak_params):
    fit_results = []

    plt.figure()
    plt.plot(x, adjusted_rate, label="Background-subtracted spectrum")

    for center, xmin, xmax in peak_params:

        mask = (x >= xmin) & (x <= xmax)
        xfit = x[mask]
        yfit = adjusted_rate[mask]

        if len(xfit) < 5:
            print(f"Fit skipped near {center}: not enough data points in range [{xmin}, {xmax}]")
            continue

        A0 = float(np.max(yfit) - np.min(yfit))
        if A0 <= 0:
            A0 = float(np.max(np.abs(yfit)))
        if A0 == 0:
            A0 = 1.0

        mu0 = float(center)
        sigma0 = max((xmax - xmin) / 6.0, 1.0)
        m0 = 0.0
        b0 = float(np.median(yfit))

        p0 = [A0, mu0, sigma0, m0, b0]

        try:
            popt, pcov = curve_fit(
                gaussian_with_background,
                xfit,
                yfit,
                p0=p0,
                maxfev=10000
            )

            A, mu, sigma, m, b = popt

            xs = np.linspace(xmin, xmax, 400)
            ys = gaussian_with_background(xs, *popt)

            plt.plot(xs, ys, linewidth=2)

            if pcov is not None:
                perr = np.sqrt(np.diag(pcov))
            else:
                perr = np.full(5, np.nan)

            fit_results.append({
                "center_ADC": float(mu),
                "center_ADC_err": float(perr[1]) if len(perr) > 1 else np.nan,
                "sigma_ADC": float(abs(sigma)),
                "sigma_ADC_err": float(perr[2]) if len(perr) > 2 else np.nan,
                "amplitude_counts_per_s": float(A),
                "background_slope": float(m),
                "background_intercept": float(b),
                "fit_xmin": float(xmin),
                "fit_xmax": float(xmax)
            })

        except Exception as e:
            print(f"Fit failed for peak near {center}: {e}")

    plt.xlabel("ADC Channel")
    plt.ylabel("Background-subtracted rate (counts/s)")
    plt.title(f"{current_rel}\nBackground-subtracted spectrum with fitted peaks")
    plt.legend()
    plt.show()

    return fit_results

In [ ]:
# -----------------------------
# FIT CURRENT FILE
# -----------------------------

fit_results_current = fit_peaks(x_current, adjusted_rate_current, peak_params_current)

print("fit_results_current:")
for row in fit_results_current:
    print(row)

In [ ]:
# -----------------------------
# FIT CURRENT FILE
# -----------------------------

fit_results_current = fit_peaks(x_current, adjusted_rate_current, peak_params_current)

print("fit_results_current:")
for row in fit_results_current:
    print(row)

In [ ]:
# -----------------------------
# SAVE RESULTS
# -----------------------------

out_dir = Path("peak_analysis_outputs")
out_dir.mkdir(exist_ok=True)

out_file = out_dir / "manual_peak_fits.csv"
results_df.to_csv(out_file, index=False)

print("Saved peak fits to:", out_file)